# Extended Data Figure 4 — Average effect size for eQTLs and cis-pQTLs

Mean absolute rescaled effect size (|β̂_rescaled|) ± 95 % CI across five variant
consequence categories for eQTLs and cis-pQTLs.

The five consequence categories (ranked by severity):

1. protein_altering
2. promoter
3. enhancer
4. intragenic
5. intergenic

**Source notebook:** `chapters/02-analysis/02-variant-effects/03_variant_regulatory_consequence.ipynb`  
**Data:** `data/intermediate_files/lead_variant_consequence_exploded/`
(pre-computed by the source notebook; re-run that notebook if missing)


## Setup


In [ ]:
from __future__ import annotations

import plotnine as p9
from gentropy.common.session import Session
from pyspark.sql import Window
from pyspark.sql import functions as f

from manuscript_methods import OpenTargetsTheme
from manuscript_methods.consequence import ConsequenceCategory
from manuscript_methods.study_statistics import StudyType

In [ ]:
session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

## Paths


In [ ]:
path_to_intermediate_data_folder = str(paper.DERIVED) + "/"

# Pre-computed by 03_variant_regulatory_consequence.ipynb
consequence_dataset_path = path_to_intermediate_data_folder + "variant_consequences"

## Load consequence dataset

This was pre-computed by `03_variant_regulatory_consequence.ipynb` and saved to parquet.


In [ ]:
consequence_dataset = session.spark.read.parquet(consequence_dataset_path)
print(f"Consequence dataset: {consequence_dataset.count():,} rows")
consequence_dataset.printSchema()

## Rank consequences per variant × study type

For each (variantId, studyType, partition) triplet, assign a severity ranking and keep the most severe consequence.


In [ ]:
w_rank = Window.partitionBy("variantId", "studyType", "partition").orderBy(f.asc("ranking"))

consequence_dataset_ranked = (
    consequence_dataset.withColumn("ranking", ConsequenceCategory.ranking(f.col("consequenceCategory")))
    .withColumn("lowestInRanking", f.dense_rank().over(w_rank))
    .withColumn("maxAbsEstimatedBeta", f.max("maxAbsEstimatedBeta").over(w_rank))
    .withColumn("con", f.size(f.collect_list("consequenceCategory").over(w_rank)))
    .dropDuplicates(["variantId", "consequenceCategory", "partition"])
    .orderBy("variantId")
)
print(f"Ranked dataset: {consequence_dataset_ranked.count():,} rows")

## Extended Data Figure 4

Filter to **eQTL** and **cis-pQTL** studies, compute mean |β| per consequence category with 95 % CI, then plot.


In [ ]:
# Filter to molecular QTL study types
w6 = Window.partitionBy("studyType", "consequenceCategory")
molqtl_studies = [StudyType.EQTL.value, StudyType.CIS_PQTL.value]

molqtl_consequence_vs_beta = (
    consequence_dataset_ranked.filter(f.col("studyType").isin(*molqtl_studies))
    .withColumn("nConsequence", f.count("variantId").over(w6))
    .withColumn("avgMaxAbsEstimatedBeta", f.avg("maxAbsEstimatedBeta").over(w6))
    .withColumn("stdMaxAbsEstimatedBeta", f.stddev("maxAbsEstimatedBeta").over(w6))
    .withColumn("seMaxAbsEstimatedBeta", f.col("stdMaxAbsEstimatedBeta") / f.sqrt(f.col("nConsequence")))
    .withColumn("CILower", f.col("avgMaxAbsEstimatedBeta") - 1.96 * f.col("seMaxAbsEstimatedBeta"))
    .withColumn("CIUpper", f.col("avgMaxAbsEstimatedBeta") + 1.96 * f.col("seMaxAbsEstimatedBeta"))
    .drop("lowestInRanking", "variantId", "maxAbsEstimatedBeta")
    .drop_duplicates(["studyType", "consequenceCategory"])
    .cache()
)
print(f"molQTL consequence vs beta: {molqtl_consequence_vs_beta.count():,}")
molqtl_consequence_vs_beta.toPandas().head()

In [ ]:
fig_ed4 = (
    molqtl_consequence_vs_beta.toPandas()
    >> p9.ggplot()
    + p9.geom_point(
        p9.aes(x="consequenceCategory", y="avgMaxAbsEstimatedBeta", color="studyType"),
        size=1.5,
        position=p9.position_dodge(width=0.3),
    )
    + p9.geom_errorbar(
        p9.aes(x="consequenceCategory", ymin="CILower", ymax="CIUpper", color="studyType"),
        width=0.3,
        position=p9.position_dodge(width=0.3),
    )
    + p9.scale_color_manual(
        values={"eqtl": "#2ca02c", "cis-pqtl": "#9467bd"},
        labels={"eqtl": "eQTL", "cis-pqtl": "cis-pQTL"},
        name="Study type",
    )
    + OpenTargetsTheme.theme
    + p9.theme(axis_text_y=p9.element_text(rotation=0))
    + p9.labs(
        x="",
        y=r"$|\hat{\beta}_{\mathrm{rescaled}}|$",
    )
    + p9.theme(legend_position="bottom", figure_size=(5, 4))
    + p9.geom_hline(p9.aes(yintercept=0), linetype="dashed", color="red", size=0.5)
    + p9.coord_flip()
)
fig_ed4
fig_ed4.save(f"{figure_dir}/extended_figure_4.pdf", dpi=300)